In [1]:
import sys
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType

spark = (SparkSession.builder
         .appName("check-python")
         .master("local[*]")
         .config("spark.pyspark.python", sys.executable)
         .config("spark.pyspark.driver.python", sys.executable)
         .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/03 15:49:33 WARN Utils: Your hostname, MacBook--Azich.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
26/03/03 15:49:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 15:49:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/03 15:49:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/03 15:49:35 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [4]:
# Read JSON file into dataframe

df = spark.read.json("zipcodes.json")
df.show(truncate=False)

+-------------------+-------+-------------+-------------------+-----+----------------------------+-----------------------+--------------+-------+-------------+------------+-----+---------------+----------+-----------+-----+-----+-----+-----------+-------+
|City               |Country|Decommisioned|EstimatedPopulation|Lat  |Location                    |LocationText           |LocationType  |Long   |Notes        |RecordNumber|State|TaxReturnsFiled|TotalWages|WorldRegion|Xaxis|Yaxis|Zaxis|ZipCodeType|Zipcode|
+-------------------+-------+-------------+-------------------+-----+----------------------------+-----------------------+--------------+-------+-------------+------------+-----+---------------+----------+-----------+-----+-----+-----+-----------+-------+
|PARC PARQUE        |US     |false        |NULL               |17.96|NA-US-PR-PARC PARQUE        |Parc Parque, PR        |NOT ACCEPTABLE|-66.22 |NULL         |1           |PR   |NULL           |NULL      |NA         |0.38 |-0.87|0.3

---
**Чтение из нескольких файлов одновременно**

Чтобы прочитать несколько файлов одновременно в DataFrame, передайте пути к файлам через запятую в метод read.json(). 

In [ ]:
# Read multiple files
df2 = spark.read.json(
    ['resources/zipcode1.json','resources/zipcode2.json'])
df2.show()  

---
**Чтение всех файлов из каталога**

Чтобы одновременно прочитать все JSON-файлы из каталога в PySpark DataFrame, используйте spark.read.json(«directory_path»), где «directory_path» указывает на каталог, содержащий JSON-файлы. PySpark автоматически обрабатывает все JSON-файлы в каталоге.

In [ ]:
# Read all JSON files from a folder
df3 = spark.read.json("/27-PySpark*.json")
df3.show()

---
**Запись PySpark DataFrame в JSON-файл**

Чтобы записать DataFrame в JSON-файл в PySpark, используйте метод write.json() и укажите путь, по которому должен быть сохранен JSON-файл. 

По желанию вы можете указать дополнительные параметры, такие как режим работы с существующими файлами и тип сжатия. Обратите внимание, что write - это объект класса DataFrameWriter.

In [ ]:
df2.write.json("zipcodes.json")

---

**Часто задаваемые вопросы по PySpark Чтение JSON**

* ***Можно ли считывать несколько файлов JSON в один DataFrame?*** 
Мы можем считывать несколько JSON-файлов в один DataFrame, указав путь к директории, содержащей JSON-файлы. PySpark автоматически объединит их в один DataFrame. Например: df = spark.read.json("path/to/json/files/")

* ***Как указать схему при чтении JSON-данных?*** 
Если вы знаете схему будущего файла и не хотите использовать стандартную опцию inferSchema, воспользуйтесь опцией schema, чтобы указать пользовательские имена столбцов и типы данных.

* ***Как работать со схемой данных JSON, содержащей вложенные структуры?***
 PySpark может обрабатывать вложенные структуры в данных JSON. Метод spark.read.json() автоматически определяет схему, включая вложенные структуры. Вы можете получить доступ к вложенным полям, используя точечную нотацию в запросах DataFrame.

* ***Что делать, если мои JSON-данные хранятся не в файле, а в переменной?*** Если данные JSON хранятся в переменной, вы можете использовать метод spark.read.json() с методом jsonRDD. Например:

In [ ]:
json_object = '{"имя": "Cynthia", "возраст": 20}'
df = spark.read.json(spark.sparkContext.parallelize([json_object]))

In [5]:
spark.stop()